<a href="https://colab.research.google.com/github/degasbr1964/Tech-Challenge-Fase-1/blob/main/Tech_Challenge_Fase_1_Edgar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install xlrd

Como melhorar o NPS das vendas online da empresa. Porque o usuário se torna detrator ou promotor da empresa.

In [6]:
# Libs - bibliotecas de exploração e manipulação de dados
import pandas as pd
import numpy as np

# Libs Gráficas
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Avisos
import warnings
warnings.filterwarnings("ignore")

In [7]:
nps = pd.read_csv("desafio_nps_fase_1.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'desafio_nps_fase_1.csv'

In [ ]:
nps.head(20)

# Tamanho da base de dados

In [ ]:
nps.shape

In [ ]:
print(f'A base de dados possui {nps.shape[0]} linhas e {nps.shape[1]} colunas')

Verificar o tipo das colunas

In [ ]:
nps.dtypes

Existem valores nulos?

In [ ]:
nps.isnull().sum().sum()

Não existem valores nulos

In [ ]:
nps.columns

Cria uma base com menos colunas

In [ ]:
df_nps = nps[['customer_age', 'customer_region',
       'customer_tenure_months', 'order_value', 'items_quantity',
       'discount_value', 'payment_installments', 'delivery_time_days',
       'delivery_delay_days', 'freight_value', 'delivery_attempts',
       'customer_service_contacts', 'resolution_time_days', 'nps_score',
       'repeat_purchase_30d', 'complaints_count', 'csat_internal_score']]

In [ ]:
df_nps.head()

##Gerando uma amostra da base inicial

In [ ]:
np.random.seed(42)
amostra = df_nps.sample(n=1000,replace=False)

In [ ]:
amostra.head()

In [ ]:
nps.describe()

In [ ]:
amostra.describe()

Altera os nomes das colunas para português

In [ ]:
df_nps.columns = ['idade', 'regiao', 'tempo_relacionamento','valor_compra','quantidade','desconto', 'parcelas','tempo_entrega', 'dias_atraso','valor_frete', 'tentativas_entrega', 'chamados_sac','tempo_resolucao','nps','recompra_30d','num_reclamacoes', 'csat']

Estatística descritiva

In [ ]:
amostra['nps_score'].hist()

In [ ]:
amostra.describe().round(2)

In [ ]:
def nps_class(nps_score):
    if nps_score <= 6:
        return "Detrator"
    elif nps_score >= 9:
        return "Promotor"
    else:
        return "Passivo"

Classificação do cliente segundo o NPS
(fonte: Bain & Company)

*   Promotores (9-10)
*   Passivos (7-8)
*   Detratores (0-6)








In [ ]:
amostra.head()

## *Função para classificar NPS*

In [ ]:
amostra["nps_class"] = amostra["nps_score"].apply(nps_class)


In [ ]:
amostra.head()

In [ ]:
df_nps['recompra_30d'].value_counts()

### Análise de médias por classificação de NPS para identificar motivos de NPS baixo

In [ ]:
nps_grouped_analysis = df_nps.groupby('classificacao_nps').mean(numeric_only=True).round(2)
display(nps_grouped_analysis.sort_values(by='nps'))

Com a tabela acima, podemos comparar as médias de diversas métricas para cada grupo de NPS. Fatores com valores significativamente maiores para 'Detrator' e menores para 'Promotor' são os potenciais maiores motivadores de um NPS baixo.

Quanto maior os dias de atraso, o tempo de resolução, os chamados ao SAC, o número de reclamações, menor o NPS e consequentemente torna esses clientes detratores da empresa.

In [ ]:
pedidos_atrasados_por_regiao = df_nps[df_nps['dias_atraso'] > 0].groupby('regiao').size().reset_index(name='pedidos_atrasados')
display(pedidos_atrasados_por_regiao)

Como podemos ver, os atrasos de entrega independem da regiaão, portanto não é um problema localizado, e sim da logística como um todo.

In [ ]:
detratores_com_atraso = df_nps[(df_nps['classificacao_nps'] == 'Detrator') & (df_nps['dias_atraso'] > 0)]
num_detratores_com_atraso = detratores_com_atraso.shape[0]
print(f"O número de Detratores que tiveram atraso na entrega é: {num_detratores_com_atraso}")

In [ ]:
detratores_com_reclamacao = df_nps[(df_nps['classificacao_nps'] == 'Detrator') & (df_nps['num_reclamacoes'] > 0)]
num_detratores_com_reclamacao = detratores_com_reclamacao.shape[0]
print(f"O número de Detratores que fizeram reclamação é: {num_detratores_com_reclamacao}")

In [ ]:
total_detratores = df_nps[df_nps['classificacao_nps'] == 'Detrator'].shape[0]
detratrores_atraso_reclamacao = df_nps[(df_nps['classificacao_nps'] == 'Detrator') & (df_nps['dias_atraso'] > 0) & (df_nps['num_reclamacoes'] > 0)]
num_detratores_atraso_reclamacao = detratrores_atraso_reclamacao.shape[0]

if total_detratores > 0:
    porcentagem_detratores_atraso_reclamacao = (num_detratores_atraso_reclamacao / total_detratores) * 100
    print(f"Do total de Detratores, {porcentagem_detratores_atraso_reclamacao:.2f}% tiveram atraso na entrega e fizeram reclamação.")
else:
    print("Não há detratores na base de dados para realizar o cálculo.")

94,54% dos clientes que tiveram atraso na entrega, fizeram reclamação ao SAC, isso nos leva a acreditar que diminuindo os problemas de logística diminuiremos os atendimentos no SAC.

In [ ]:
detratores_reclamacao_sem_atraso = df_nps[(df_nps['classificacao_nps'] == 'Detrator') & (df_nps['num_reclamacoes'] > 0) & (df_nps['dias_atraso'] == 0)]
num_detratores_reclamacao_sem_atraso = detratores_reclamacao_sem_atraso.shape[0]
print(f"O número de Detratores que fizeram reclamação e não tiveram atraso na entrega é: {num_detratores_reclamacao_sem_atraso}")

Os detratores que não tiveram problemas na entrega é pequeno, e pode ser tratado por um atendimento personalizado para descobrir o que motivou isso.

In [ ]:
promotores_com_atraso = df_nps[(df_nps['classificacao_nps'] == 'Promotor') & (df_nps['dias_atraso'] > 0)]
num_promotores_com_atraso = promotores_com_atraso.shape[0]
print(f"O número de Promotores que tiveram atraso na entrega é: {num_promotores_com_atraso}")